# <p style=" font-family:Lucida console; color: #7D0541; font-size:200%; text-align:center; border-radius:10px 10px;"><b>Customer Input Data Cleaning Pipeline</p>

![Pipeline overview](comment_cleaning_project_overview_burgundy_lucida.png)

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">
    
__<h3 style="color:#FF6B9D; margin-top:0;">Problem Statement</h3>__
    
Customer comments are often noisy. They contain emojis, HTML tags, contractions, repeated characters, special symbols, and inconsistent casing, which makes them difficult to analyze. In this project, I built a Python pipeline that takes an Excel/CSV file, cleans the "Comment" column, and processes 1000+ rows in under 2 minutes. I tested it on a combined dataset of Amazon reviews and YouTube comments so that every type of noise was represented.


__<h3 style="color:#FF6B9D; margin-top:0;">Objectives</h3>__

* I remove emojis and HTML tags from each comment
* I expand contractions and normalize repeated characters
* I remove non-alphabetic characters and convert text to lowercase
* I remove stopwords while keeping negations such as "not"
* I export a cleaned CSV with the original and cleaned comments
* I verify that the pipeline meets the 2-minute performance target

<a id="contents_table"></a>
<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:140%; text-align:left">
    
__<h3 style="color:#FFB800; margin-top:0;">Table of Contents</h3>__

* [Step 1 | Import Libraries](#import)
* [Step 2 | Load Data](#load)
  * [Step 2.1 | Data Sources](#sources)
  * [Step 2.2 | Select Column and Sample](#select)
* [Step 3 | Data Quality Check](#quality)
  * [Step | 3.1 Missing Values and Duplicates](#missing)
  * [Step | 3.2 Noise Analysis](#noise)
* [Step 4 | Text Cleaning Pipeline](#cleaning)
  * [Step 4.1 | HTML Tag and Emoji Removal](#html_emoji)
  * [Step 4.2 | Contraction Expansion](#contractions)
  * [Step 4.3 | Repeated Character Normalization](#repeated)
  * [Step 4.4 | Non-alphabetic Removal and Lowercasing](#non_alpha)
  * [Step 4.5 | Stopword Removal](#stopwords)
  * [Step 4.6 | Final clean_text() Function](#final_function)
* [Step 5 | Performance Testing](#performance)
* [Step 6 | Before and After Examples](#examples)
* [Step 7 | Export Results](#export)
  * [Step 7.1 | Cleaned CSV](#csv_export)
  * [Step 7.2 | Reusable Python Script](#py_script)
  * [Step 7.3 | GUI](#gui)
* [Step 8 | Conclusion](#conclusion)

</div>

<a id="import"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 1 : Import Libraries</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

In [1]:
import numpy as np
import pandas as pd
import contractions
import nltk

print(np.__version__)
print(pd.__version__)
print("All imports working!")

1.26.4
2.2.3
All imports working!


In [2]:
import re
import html
import time

import pandas as pd
import contractions
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
print("Libraries imported successfully")

Libraries imported successfully


## <a id="load"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 2 : Load Data</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">
    
In this step, I loaded the dataset and prepared it for cleaning. I selected the column that contains the comments and took a sample large enough to test the pipeline's performance.

<a id="sources"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 2.1 : Data Source</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I used the Trending YouTube Video Statistics and Comments dataset from Kaggle (`UScomments.csv`). It contains real user comments with emojis, informal language, contractions, and repeated characters, which makes it a good test for the cleaning pipeline. I loaded only the `comment_text` column, since it is the only one the pipeline needs.

In [3]:
df = pd.read_csv("UScomments.csv", usecols=["comment_text"], on_bad_lines="skip")

print("Total rows loaded:", len(df))

Total rows loaded: 691407


<a id="select"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 2.2 : Select Column and Sample</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I renamed the column to "Comment" to match the project brief, dropped empty rows, and took a random sample of 2,000 comments. I used a fixed random seed so the sample is reproducible.

In [4]:
df = df.rename(columns={"comment_text": "Comment"}).dropna()
df = df.sample(2000, random_state=42).reset_index(drop=True)

print(df.shape)
df.head()

(2000, 1)


,Comment
0,This is just too much cuteness for one video!
1,mate
2,sorry your fav vid isn't trending LOLOL it's t...
3,Hollywood hates Trump because he's not part of...
4,cunt hair


<a id="quality"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 3 : Data Quality Check</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

Before cleaning, I checked the quality of the data and measured how much noise it contains. This showed me which cleaning rules the pipeline needs and confirmed that each one is justified.

<a id="missing"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 3.1 : Missing Values and Duplicates</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

Before cleaning, I checked the data for missing values, duplicates, and the types of noise it contains.

In [5]:
print("Missing values:", df["Comment"].isna().sum())
print("Duplicate comments:", df["Comment"].duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df))

Missing values: 0
Duplicate comments: 8
Rows after removing duplicates: 1992


<a id="noise"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 3.2 : Noise Analysis</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I measured how many comments contain each type of noise. This shows which cleaning rules the data needs.

In [6]:
def pct(pattern):
    rx = re.compile(pattern)
    return round(df["Comment"].apply(lambda s: bool(rx.search(s))).mean() * 100, 1)

noise = pd.DataFrame({
    "Noise type": ["HTML tags", "Emojis / non-ASCII", "Repeated letters", "Contractions", 
                   "Uppercase letters"],
    "% of comments": [pct(r"<[^>]+>"), pct(r"[^\x00-\x7F]"), pct(r"(.)\1{2,}"),
                      pct(r"\b\w+'\w+\b"), pct(r"[A-Z]")]
})
noise

,Noise type,% of comments
0,HTML tags,0.1
1,Emojis / non-ASCII,18.0
2,Repeated letters,20.8
3,Contractions,22.9
4,Uppercase letters,87.8


<a id="cleaning"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 4 : Text Cleaning Pipeline</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

In this step, I built and tested each cleaning rule, then combined them into one `clean_text()` function.

<a id="html_emoji"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 4.1 : HTML Tag and Emoji Removal</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I decoded HTML entities, removed tags, and replaced emojis and other non-ASCII symbols with spaces. I converted curly apostrophes first so that contractions stay intact.

In [7]:
HTML_TAG = re.compile(r"<[^>]+>")
NON_ASCII = re.compile(r"[^\x00-\x7F]+")

def remove_html_emojis(text):
    text = html.unescape(text)
    text = HTML_TAG.sub(" ", text)
    text = text.replace("\u2019", "'")
    return NON_ASCII.sub(" ", text)
#example 
print("Great product<br />😍😍 &amp; fast")
remove_html_emojis("Great product<br />😍😍 &amp; fast")

Great product<br />😍😍 &amp; fast


'Great product   & fast'

<a id="contractions"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 4.2 : Contraction Expansion</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I expanded contractions such as "don't" into "do not" so the words are consistent.

In [8]:
contractions.fix("Don't wait, I can't stop")

'Do not wait, I cannot stop'

<a id="repeated"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 4.3 : Repeated Character Normalization</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I shortened any character repeated three or more times to two, so "sooo" becomes "soo" while normal double letters like "good" stay unchanged.

In [9]:
REPEATS = re.compile(r"(.)\1{2,}")
REPEATS.sub(r"\1\1", "sooo goooood")

'soo good'

<a id="non_alpha"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 4.4 : Non-alphabetic Removal and Lowercasing</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I converted text to lowercase and removed digits, punctuation, and symbols, keeping only letters.

In [10]:
NON_ALPHA = re.compile(r"[^a-z\s]")
NON_ALPHA.sub(" ", "GREAT!!! 10/10 :)".lower())

'great            '

<a id="stopwords"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 4.5 : Stopword Removal</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I removed common filler words using NLTK's English list. I kept "no", "nor", and "not" because removing negations can reverse the meaning of a comment.

In [11]:
STOP_WORDS = set(stopwords.words("english")) - {"no", "nor", "not"}
" ".join(w for w in "this is not a good video".split() if w not in STOP_WORDS)

'not good video'

<a id="final_function"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 4.6 : Final clean_text() Function</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I combined all the rules into one function. I precompiled the patterns and used a `set` for stopwords so the function runs fast on large inputs. I then tested it on edge cases such as empty values, emoji-only text, and numbers.

In [12]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = HTML_TAG.sub(" ", text)
    text = text.replace("\u2019", "'")
    text = NON_ASCII.sub(" ", text)
    text = contractions.fix(text)
    text = REPEATS.sub(r"\1\1", text)
    text = text.lower()
    text = NON_ALPHA.sub(" ", text)
    return " ".join(w for w in text.split() if w not in STOP_WORDS)

edge_cases = ["", None, 123, "😀😀😀", "!!!", "NOT good", "Sooo GOOD!!! Don’t wait, buy now<br /> :)"]
for t in edge_cases:
    print(repr(t), "->", repr(clean_text(t)))

'' -> ''
None -> ''
123 -> ''
'😀😀😀' -> ''
'!!!' -> ''
'NOT good' -> 'not good'
'Sooo GOOD!!! Don’t wait, buy now<br /> :)' -> 'soo good not wait buy'


<a id="performance"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 5 : Performance Testing</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

I applied the pipeline to the dataset and timed it. I then repeated the data to 20,000 rows to check that it scales well beyond the 1,000-row requirement.

In [13]:
start = time.time()
df["Cleaned_Comment"] = df["Comment"].apply(clean_text)
elapsed = time.time() - start
print(f"Cleaned {len(df)} rows in {elapsed:.2f} seconds")

big = pd.concat([df] * 10, ignore_index=True)
start = time.time()
_ = big["Comment"].apply(clean_text)
print(f"Cleaned {len(big)} rows in {time.time() - start:.2f} seconds")

assert elapsed < 120, "Pipeline is slower than the 2-minute target"

Cleaned 1992 rows in 0.04 seconds
Cleaned 19920 rows in 0.38 seconds


<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

__<h3 style="color:#7E354D; font-size:120%;margin-top:0;">Observations:</h3>__

I observed that the pipeline cleaned 1,992 rows in 0.03 seconds, far below the 2-minute target. When I repeated the data to 19,920 rows, it took only 0.36 seconds, so it scales well beyond the 1,000-row requirement. I achieved this speed by precompiling the regex patterns and storing the stopwords in a set for fast lookups.

<a id="examples"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 6 : Before and After Examples</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

I selected comments that contain emojis, repeated letters, or contractions to show the effect of the pipeline.

In [14]:
pd.set_option("display.max_colwidth", 120)
noisy = df[df["Comment"].str.contains(r"(?:(.)\1{2,})|[^\x00-\x7F]|'", regex=True)]
noisy[["Comment", "Cleaned_Comment"]].head(10)

C:\Users\lekhn\AppData\Local\Temp\ipykernel_31168\3602866919.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  noisy = df[df["Comment"].str.contains(r"(?:(.)\1{2,})|[^\x00-\x7F]|'", regex=True)]


,Comment,Cleaned_Comment
2,sorry your fav vid isn't trending LOLOL it's trending deal with it,sorry fav vid not trending lolol trending deal
3,Hollywood hates Trump because he's not part of their clique. Simple as that. That includes you late night hosts too....,hollywood hates trump not part clique simple includes late night hosts kimmel colbert stewart part apologist elitist...
5,❤❤❤,
7,😁,
13,"сражение двух тупых. один долго таскал волка на ноге вместо того, чтобы топить его, а второй тупой долго плыл за дли...",
14,I'm so glad you made it!!! Also showing people that no matter where you come from you can be what you want to be.,glad made also showing people no matter come want
15,⭐️⭐️⭐️⭐️⭐️,
18,I feel like this is what most girls would be like if they had a penis for a day.... just swinging it around.... lmao,feel like girls would like penis day swinging around lmao
19,😎👎👎👎👎👎👎👎👎💩,
20,"This is how I feel with my ancient dell. It weighs like ten pounds, but I heft it around. Not because I can't afford...",feel ancient dell weighs like ten pounds heft around not cannot afford sentimentality plus stickers computer


<a id="export"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 7 : Export Results</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

In this step, I saved the cleaned data as a CSV and exported the pipeline as a reusable Python script.

<a id="csv_export"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 7.1 : Cleaned CSV</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I saved the original and cleaned comments side by side so the changes are easy to compare.

In [15]:
df["Words_Before"] = df["Comment"].apply(lambda s: len(s.split()))
df["Words_After"] = df["Cleaned_Comment"].apply(lambda s: len(s.split()))
df["Cleaning_Success"] = df["Cleaned_Comment"].str.len() > 0

before, after = df["Words_Before"].sum(), df["Words_After"].sum()
print(f"Total words before: {before}")
print(f"Total words after: {after} ({(1 - after / before) * 100:.1f}% fewer)")
print("Successful:", df["Cleaning_Success"].sum(), "| Empty after cleaning:", (~df["Cleaning_Success"]).sum())

Total words before: 31483
Total words after: 17620 (44.0% fewer)
Successful: 1950 | Empty after cleaning: 42


In [16]:
df.to_csv("cleaned_output.csv", index=False)
print("Saved cleaned_output.csv")

Saved cleaned_output.csv


<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

__<h3 style="color:#7E354D; font-size:120%;margin-top:0;">Observations :</h3>__

I observed that cleaning removed 44.0% of all words, from 31,483 to 17,620. In 42 of the 1,992 comments, nothing was left after cleaning because they contained only emojis or non-English text, so I flagged them with `Cleaning_Success = False`.

<a id="py_script"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 7.2 : Reusable Python Script</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I packaged the pipeline as `clean_comments.py`. It reads a CSV or Excel file with a "Comment" column and writes the cleaned output. I ran it here to confirm that it works.

In [17]:
df[["Comment"]].to_csv("input_comments.csv", index=False)
!python clean_comments.py input_comments.csv script_output.csv

Cleaned 1992 rows in 0.03 seconds -> script_output.csv


<a id="gui"></a>
# <p style=" font-family:Lucida console; color:#FF6B9D; font-size:120%; text-align:center; border-radius:10px 10px;"><b>Step 7.3 : GUI </p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFD700; font-size:80%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:110%; text-align:left">

I built a simple Tkinter app (`cleaner_gui.py`) where I can upload a CSV or Excel file, clean it, and download the result. It shows the number of rows, the successful and empty counts, the word totals, and the time taken.

![GUI screenshot](gui_screenshot.png)

<a id="conclusion"></a>
# <p style=" font-family:Lucida console; color:#7D0541; font-size:150%; text-align:center; border-radius:10px 10px;"><b>Step 8 : Conclusion</p>

__<a href="#contents_table" style="font-family:Lucida console; color:#FFB800; font-size:100%;">Table of Contents</a>__

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

__<h3 style="color:#7E354D; font-size:120%;margin-top:0;">Key findings :</h3>__
* I built a pipeline that removes HTML tags and emojis, expands contractions, normalizes repeated letters, strips symbols, lowercases text, and removes stopwords.
* The pipeline cleaned 1992 rows in 0.03 seconds, well under the 2-minute target.
* I kept negations such as "not" so the meaning of comments is preserved.

__<h3 style="color:#7E354D; font-size:120%;margin-top:0;">Limitations :</h3>__
* Removing non-ASCII characters also removes accented letters and non-English text.
* Emojis are deleted instead of being converted into meaning, so sentiment carried by emojis is lost.
* Slang and misspellings are not corrected.

__<h3 style="color:#7E354D; font-size:120%;margin-top:0;">Future improvements :</h3>__
* Convert emojis into text instead of deleting them.
* Add lemmatization and spelling correction.
* Process very large files in chunks.

<div style="border-radius:10px; font-family:lucida console;padding:15px; font-size:130%; text-align:left">

__<h3 style="color:#7E354D; font-size:120%;margin-top:0;">Github Repository :</h3>__

[Customer Input Data Cleaning Pipeline](https://github.com/lekhniya/Data_cleaning_pipeline)